In [ ]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
# src/ouput_schemas/

In [ ]:
class CodeIssue(BaseModel):
    """A specific issue or suggestion found in the code."""
    line_number: int | None = Field(
        default=None,
        description="Approximate line number where the issue was found, if applicable.",
    )
    severity: Literal["critical", "warning", "info"] = Field(
        description="Severity level of the issue.",
    )
    category: Literal["bug", "security", "performance", "style", "maintainability"] = Field(
        description="Category classification of the issue.",
    )
    description: str = Field(
        description="Detailed explanation of the issue or concern.",
    )
    suggestion: str = Field(
        description="Actionable suggestion or recommendation to fix/improve it.",
    )


In [ ]:
class ReviewResult(BaseModel):
    summary: str = Field(
        description="High-level overview of code quality, strengths, and primary concerns.",
    )
    score: int = Field(
        description="Overall quality score from 1 (poor) to 10 (exemplary).",
        ge=1,
        le=10,
    )
    issues: list[CodeIssue] = Field(
        default_factory=list,
        description="List of detected code issues or improvement points.",
    )
    refactored_code: str | None = Field(
        default=None,
        description="Optional full refactored version of the code incorporating fixes, if appropriate.",
    )

In [ ]:
class TestCase(BaseModel):
    """An individual test case specification."""
    name: str = Field(
        description="Identifier name of the test function (e.g. 'test_valid_login').",
    )
    description: str = Field(
        description="Description of what this test case verifies (e.g. edge cases, happy path).",
    )
    target_function: str = Field(
        description="Name of the specific function or class being tested.",
    )

In [ ]:
class TestSuite(BaseModel):
    """Generated test suite containing complete runnable tests."""
    framework: str = Field(
        default="pytest",
        description="Testing framework used (e.g. 'pytest', 'unittest').",
    )
    test_code: str = Field(
        description="Complete, valid, ready-to-run Python test file contents.",
    )
    test_cases: list[TestCase] = Field(
        default_factory=list,
        description="List of test cases included in the suite.",
    )


In [ ]:
# src/states/

In [ ]:
class State:
    code: str
    language: str
    file_path: str|None
    review: ReviewResult|None
    tests: TestSuite|None
    error: str|None

In [ ]:
# src/prompts/code_review_nodes.py

In [ ]:
"""Prompts for P3 Code Review graph nodes."""

_REVIEW_SYSTEM_PROMPT = """You are a Principal Software Engineer and Code Reviewer.
Your task is to conduct a thorough, rigorous, and actionable code review of the provided code snippet or file.

Review Dimensions:
1. Correctness & Bugs: Logic flaws, off-by-one errors, unhandled exceptions, race conditions, edge-case failures.
2. Security: Injection risks, improper validation, secret leakage, unsafe operations.
3. Performance: Inefficient algorithms, memory leaks, unnecessary I/O, N+1 queries.
4. Maintainability & Style: Modularity, clean architecture, readability, idiomatic conventions (e.g. PEP 8 for Python), typing accuracy.

Guidelines:
- Be specific and cite line numbers where possible.
- Provide clear, constructive, and actionable fix suggestions.
- Rate the overall code quality objectively on a scale of 1 to 10.
- If significant improvements are needed, provide a clean, complete refactored version of the code.
"""

REVIEW_PROMPT = ChatPromptTemplate.from_messages(
    [
        ("system", _REVIEW_SYSTEM_PROMPT),
        (
            "human",
            "Please review the following {language} code:\n"
            "File path: {file_path}\n\n"
            "```\n{code}\n```",
        ),
    ]
)

In [ ]:
"""Prompts for P3 Test Generation graph nodes."""

_TEST_GEN_SYSTEM_PROMPT = """You are a Senior QA Automation Engineer specializing in unit and integration testing.
Your task is to generate a comprehensive, robust, and self-contained test suite for the provided code.

Testing Standards:
1. Coverage: Happy paths, edge cases, boundary values, invalid inputs, and error/exception handling.
2. Structure: Follow {framework} conventions with clear test names (`test_<scenario>_<expected_behavior>`).
3. Isolation & Independence: Tests must be deterministic and runnable without external side effects (mock network/disk if needed).
4. Completeness: The `test_code` field must contain a complete, syntactically valid, ready-to-run test file including all necessary imports.

If code review findings are provided, write specific regression tests covering the identified issues.
"""

TEST_GEN_PROMPT = ChatPromptTemplate.from_messages(
    [
        ("system", _TEST_GEN_SYSTEM_PROMPT),
        (
            "human",
            "Generate a complete {framework} test suite for the following {language} code.\n"
            "File path: {file_path}\n\n"
            "Source Code:\n```\n{code}\n```\n\n"
            "Review Summary & Findings:\n{review_summary}",
        ),
    ]
)

In [ ]:
from typing import Any
from pydantic import BaseModel
from langchain_core.language_models import BaseChatModel
from langchain_core.prompts import BasePromptTemplate
from utils.get_text import get_text


def generate_response(
    prompt: BasePromptTemplate,
    model: BaseChatModel,
    inputs: dict[str, Any],
    schema: type[BaseModel] | None = None,
) -> Any:
    """Execute prompt + model chain with optional structured output parsing.

    - If schema is provided: binds schema and returns the parsed Pydantic object.
    - If schema is None: executes as plain text and returns str via get_text.
    """
    if schema is not None:
        structured_llm = model.with_structured_output(schema)
        chain = prompt | structured_llm
        return chain.invoke(inputs)

    chain = prompt | model
    response = chain.invoke(inputs)
    return get_text(response)

In [ ]:
from alireza.py_files.model import gemini_flashlite
from alireza.py_files.state import CodeReviewState


def review_node(state: CodeReviewState) -> ReviewResult | None:
    model = gemini_flashlite
    prompt = REVIEW_PROMPT
    inputs = {
        "code": state["code"], 
        "language": state["language"], 
        "file_path": state["file_path"],
    }
    schema = ReviewResult
    
    response = generate_response(prompt, model, inputs, schema)

In [ ]:
import uuid
from langchain_core.messages import AIMessage
from langchain_core.runnables import RunnableLambda

from graphs.code_review_parentgraph import build_code_review_graph
from utils.get_text import get_text

In [ ]:
_code_review_graph = build_code_review_graph()
def _run_code_review_subagent(state: dict) -> dict:
  """Adapter function: converts incoming message to CodeReviewState and returns formatted AIMessage."""
  input_text = get_text(state["messages"][-1])
  config = {"configurable": {"thread_id": str(uuid.uuid4())}}

  result = _code_review_graph.invoke(
      {
          "code": input_text,
          "language": "python",
          "file_path": None,
      },
      config=config,
  )

  parts = []
  review = result.get("review")
  if review:
    parts.append(f"### Code Review Summary (Score: {review.score}/10)")
    parts.append(review.summary)
    if review.issues:
      parts.append("\n**Key Issues Found:**")
      for issue in review.issues:
        loc = (
            f"Line {issue.line_number}: "
            if issue.line_number is not None
            else ""
        )
        parts.append(f"- [{issue.severity.upper()}] {loc}{issue.description}")

  tests = result.get("tests")
  execution = result.get("test_execution")
  if tests:
    status_str = (
        "PASSED" if (execution and execution.passed) else "FAILED / PENDING"
    )
    parts.append(
        f"\n### Test Suite Status: {status_str} ({len(tests.test_cases)} test"
        " cases)"
    )

  applied_code = result.get("applied_code")
  if applied_code:
    parts.append(f"\n### Final Code Output:\n```python\n{applied_code}\n```")

  output_content = (
      "\n".join(parts) if parts else "Code review completed with no output."
  )
  return {"messages": [AIMessage(content=output_content)]}

In [ ]:
code_review_subagent = {
    "name": "code_review_subagent",
    "description": (
        """Analyzes and reviews source code, identifies bugs, security vulnerabilities,
        and performance flaws,
        generates and executes pytest test suites in an isolated sandbox,
        and outputs refactored code."""
    ),
    "runnable": RunnableLambda(_run_code_review_subagent)
}

In [ ]:
from pprint import pprint
from langchain_core.messages import HumanMessage
from subagents import code_review_subagent
test_code = '''
    def find_maximum(numbers: list[int]) -> int:
        max_val = 0
        for n in numbers:
            if n > max_val:
                max_val = n
        return max_val
    '''

config = {"configurable": {"thread_id": "test-run-2"}}
print('🤖 Invoking code_review_subagent as a Coordinator...\n')
response = code_review_subagent['runnable'].invoke({
    'messages': [HumanMessage(content=test_code)],
    },
    config=config,
)

pprint(response['messages'][0].content)

In [1]:
from langchain_core.messages import HumanMessage
from subagents import code_review_subagent


In [2]:
def _run_code_review_subagent(state: dict) -> dict:
    input_text = get_text(state["messages"][-1])
    thread_id = str(uuid.uuid4())
    config = {"configurable": {"thread_id": thread_id}}
    _code_review_graph = build_code_review_graph()

    result = _code_review_graph.invoke(
        {"code": input_text, "language": "python", "file_path": None},
        config=config,
    )


    parts = []
    review = result.get("review")
    if review:
        parts.append(f"### Code Review Summary (Score: {review.score}/10)")
        parts.append(review.summary)
        if review.issues:
            parts.append("\n**Key Issues Found:**")
            for issue in review.issues:
                loc = (
                    f"Line {issue.line_number}: "
                    if issue.line_number is not None
                    else ""
                )
                parts.append(f"- [{issue.severity.upper()}] {loc}{issue.description}")

    tests = result.get("tests")
    execution = result.get("test_execution")
    if tests:
        status_str = (
            "PASSED" if (execution and execution.passed) else "FAILED / PENDING"
        )
        parts.append(
            f"\n### Test Suite Status: {status_str} ({len(tests.test_cases)} test"
            " cases)"
        )
    refactored_code = result.get("applied_code") or (
        review.refactored_code if review else None
    )
    if refactored_code:
        parts.append(
            f"\n### Suggested Refactored Code:\n```python\n{refactored_code}\n```"
        )

    output_content = (
        "\n".join(parts) if parts else "Code review completed with no output."
    )
    return {
        "messages": [AIMessage(content=output_content)],
        "thread_id": thread_id
    }

In [3]:
test_code = '''
    def find_maximum(numbers: list[int]) -> int:
        max_val = 0
        for n in numbers:
            if n > max_val:
                max_val = n
        return max_val
    '''

response = code_review_subagent["runnable"].invoke({
    "messages": [HumanMessage(content=test_code)]
})

thread_id = response["thread_id"]
config = {"configurable": {"thread_id": thread_id}}

history = list(_code_review_graph.get_state_history(config))
for snap in history:
    print(snap.metadata.get("step"), snap.next)

KeyboardInterrupt: 